# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [1]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = train["daily_orders"].to_numpy(float)

QUANTILES = (0.1, 0.9)

In [3]:
train.head()

,id,city,district_type,latitude,longitude,dist_to_center_km,premise_area_sqm,fitting_rooms_count,entrance_type,floor,parking_nearby,dist_to_metro_m,dist_to_bus_stop_m,residential_units_500m,residential_units_1000m,competitor_pvz_count_500m,commercial_poi_count_500m,avg_housing_price_sqm,nearest_wb_pvz_dist_m,daily_orders
0,PVZ_TR_000001,Самара,center,53.24416,50.17513,3.08,72.5,4,courtyard,-1,1,3122,15,651,1803,1,9,146740.0,444.0,42
1,PVZ_TR_000002,Новосибирск,suburb,54.99110,82.86342,5.00,98.7,6,shopping_mall,1,0,3246,15,490,1268,1,6,144137.0,389.0,49
2,PVZ_TR_000003,Москва,industrial,55.58814,37.47537,20.66,62.3,3,courtyard,-1,1,5000,129,131,368,0,4,277898.0,802.0,25
3,PVZ_TR_000004,Москва,industrial,55.59576,37.45754,20.42,71.7,4,shopping_mall,1,1,5000,377,160,404,1,0,175257.0,1072.0,25
4,PVZ_TR_000005,Екатеринбург,suburb,56.90393,60.50738,9.38,90.8,5,courtyard,-1,1,5000,36,394,1133,1,8,125757.0,250.0,25


In [4]:
train.describe(include='all')

,id,city,district_type,latitude,longitude,dist_to_center_km,premise_area_sqm,fitting_rooms_count,entrance_type,floor,parking_nearby,dist_to_metro_m,dist_to_bus_stop_m,residential_units_500m,residential_units_1000m,competitor_pvz_count_500m,commercial_poi_count_500m,avg_housing_price_sqm,nearest_wb_pvz_dist_m,daily_orders
count,7932,7932,7932,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932.000000,7932.00000,7932.000000
unique,7932,8,5,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,PVZ_TR_007932,Москва,suburb,NaN,NaN,NaN,NaN,NaN,street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,2809,4284,NaN,NaN,NaN,NaN,NaN,4376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,55.905835,44.385471,10.555116,83.597138,4.744957,NaN,0.710792,0.621659,4322.376702,180.841528,635.417423,1857.878215,2.141957,6.374307,215661.142083,718.61939,91.568205
std,NaN,NaN,NaN,3.258728,14.731240,5.472248,29.552035,1.863081,NaN,0.839733,0.485004,1962.248777,164.516307,756.551825,2441.785213,1.517691,4.735864,84075.362957,457.77870,115.799551
min,NaN,NaN,NaN,45.000230,30.150150,0.080000,32.000000,2.000000,NaN,-1.000000,0.000000,40.000000,15.000000,58.000000,158.000000,0.000000,0.000000,70000.000000,10.00000,25.000000
25%,NaN,NaN,NaN,55.580690,37.461160,6.490000,57.800000,3.000000,NaN,1.000000,0.000000,3148.000000,51.000000,183.500000,488.750000,1.000000,3.000000,150515.500000,378.00000,25.000000
50%,NaN,NaN,NaN,55.759855,37.789685,9.770000,83.650000,5.000000,NaN,1.000000,1.000000,5000.000000,127.000000,387.000000,1031.000000,2.000000,5.000000,194416.500000,619.00000,48.000000
75%,NaN,NaN,NaN,56.858717,49.215600,13.970000,109.000000,6.000000,NaN,1.000000,1.000000,5000.000000,260.000000,612.250000,1731.250000,3.000000,9.000000,268543.250000,962.00000,96.000000


In [5]:
te_id = test['id']
tr = train.drop(columns=['id', 'daily_orders'])
te = test.drop(columns=['id'])

catf = ['city', 'district_type', 'entrance_type']

In [6]:
tr.head()

,city,district_type,latitude,longitude,dist_to_center_km,premise_area_sqm,fitting_rooms_count,entrance_type,floor,parking_nearby,dist_to_metro_m,dist_to_bus_stop_m,residential_units_500m,residential_units_1000m,competitor_pvz_count_500m,commercial_poi_count_500m,avg_housing_price_sqm,nearest_wb_pvz_dist_m
0,Самара,center,53.24416,50.17513,3.08,72.5,4,courtyard,-1,1,3122,15,651,1803,1,9,146740.0,444.0
1,Новосибирск,suburb,54.99110,82.86342,5.00,98.7,6,shopping_mall,1,0,3246,15,490,1268,1,6,144137.0,389.0
2,Москва,industrial,55.58814,37.47537,20.66,62.3,3,courtyard,-1,1,5000,129,131,368,0,4,277898.0,802.0
3,Москва,industrial,55.59576,37.45754,20.42,71.7,4,shopping_mall,1,1,5000,377,160,404,1,0,175257.0,1072.0
4,Екатеринбург,suburb,56.90393,60.50738,9.38,90.8,5,courtyard,-1,1,5000,36,394,1133,1,8,125757.0,250.0


In [13]:
def predict(x_train, y_train, x_test, catf):
    model = CatBoostRegressor(iterations=1000, cat_features=catf, verbose=200)
    model.fit(x_train, y_train)
    q = np.quantile(y_train, QUANTILES)
    return np.array([[q[0]] * len(x_test), list(model.predict(x_test)), [q[1]] * len(x_test)]).T

In [14]:
# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(tr.iloc[a], y[a], tr.iloc[b], catf)
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

Learning rate set to 0.054268
0:	learn: 107.3920105	total: 8.18ms	remaining: 8.17s
200:	learn: 22.4039465	total: 1.22s	remaining: 4.85s
400:	learn: 18.1975680	total: 2.16s	remaining: 3.23s
600:	learn: 15.8039542	total: 3.08s	remaining: 2.04s
800:	learn: 13.9942055	total: 4.06s	remaining: 1.01s
999:	learn: 12.4502971	total: 5.42s	remaining: 0us
Learning rate set to 0.054268
0:	learn: 113.9627028	total: 3.33ms	remaining: 3.33s
200:	learn: 24.2183545	total: 1.01s	remaining: 4.01s
400:	learn: 19.2934870	total: 2.09s	remaining: 3.12s
600:	learn: 16.5812398	total: 3.44s	remaining: 2.29s
800:	learn: 14.3551889	total: 4.69s	remaining: 1.17s
999:	learn: 12.8374805	total: 6.35s	remaining: 0us
Learning rate set to 0.054268
0:	learn: 114.6269594	total: 4.26ms	remaining: 4.26s
200:	learn: 24.6747160	total: 1.45s	remaining: 5.74s
400:	learn: 20.1668153	total: 2.4s	remaining: 3.59s
600:	learn: 16.8927575	total: 3.36s	remaining: 2.23s
800:	learn: 14.6658378	total: 5.03s	remaining: 1.25s
999:	learn: 12

In [15]:
# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test_y = np.round(predict(tr, y, te, catf), 1)
sub = pd.DataFrame({"id": te_id,
                    "predicted_daily_orders_p10": test_y[:, 0],
                    "predicted_daily_orders_p50": test_y[:, 1],
                    "predicted_daily_orders_p90": test_y[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("catboost + constant", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости

Learning rate set to 0.056791
0:	learn: 110.8747619	total: 8.72ms	remaining: 8.71s
200:	learn: 23.9991679	total: 1.66s	remaining: 6.62s
400:	learn: 19.9106561	total: 3.09s	remaining: 4.62s
600:	learn: 17.3800735	total: 4.35s	remaining: 2.89s
800:	learn: 15.5224645	total: 5.45s	remaining: 1.35s
999:	learn: 13.9479342	total: 6.5s	remaining: 0us
{"name": "catboost + constant", "metric": "quantile_quality_x100", "folds": [30.71, 31.8, 30.63, 30.75], "cv": 30.97, "std": 0.48}
